# L05 · 현대 OPD from scratch

## Goal

**예상 시간:** 50분 · **경로:** fast, full

- student rollout부터 업데이트까지 잇는다
- full/sampled estimator를 구분한다
- teacher gradient를 차단한다

### 현재 위치: L04 → **L05** → L06

```text
Prompt/Data -> state source -> ... -> L05 -> ... -> fair evaluation
```

Alt text: The course map highlights L05 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L05"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L05', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

현대 sampled-token OPD에서 sampling은 미분되지 않는다. rollout log-prob은 snapshot이고, update 시 student logits을 다시 계산해야 현재 parameter로 gradient가 흐른다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

한 update는 `collect → freeze trajectory → teacher score → recompute student logits → masked loss → optimizer step` 순서다. rollout token은 discrete sample이라 그 선택을 통해 미분하지 않는다. rollout-time log-prob은 감사/importance 정보이고, 실제 gradient는 같은 token sequence를 현재 student에 다시 넣어 얻은 log-prob에서 나온다.

full reverse KL은 모든 vocabulary 항을 합해 낮은 분산의 정확한 token-state objective를 준다. sampled reverse KL은 `y ~ student`에서 `(log p_student(y)-log p_teacher(y)).detach() * log p_student(y)`를 사용한다. 메모리는 작지만 분산과 baseline 설계 문제가 생긴다.

### 실제 구현: 왜 이렇게 만들었나

`collect_student_trajectories`는 generation 동안 student의 기존 train/eval mode를 복원하고 log-prob snapshot을 detach한다. loss는 response target 위치만 한 칸 shift해 계산한다. teacher/student logits shape가 다르면 forward 전에 실패한다.

실제 코드: [`rollout.py`](../../src/opd_study/algorithms/rollout.py), [`losses.py`](../../src/opd_study/algorithms/losses.py).

In [2]:
import inspect
from opd_study.algorithms import collect_student_trajectories, sampled_reverse_kl_loss

objects_to_show = (collect_student_trajectories, sampled_reverse_kl_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.rollout.collect_student_trajectories
def collect_student_trajectories(
    student: TinyCausalLM,
    prompts: Sequence[str],
    tokenizer: CharacterTokenizer,
    *,
    max_new_tokens: int = 64,
    min_new_tokens: int = 0,
    temperature: float = 1.0,
    generator: torch.Generator | None = None,
) -> TrajectoryBatch:
    """Sample responses and save detached rollout-time selected log-probabilities."""

    if not prompts:
        raise ValueError("prompts must not be empty")
    device = _module_device(student)
    sequences: list[Tensor] = []
    prompt_lengths: list[int] = []
    for prompt in prompts:
        prompt_tensor = torch.tensor(
            tokenizer.encode(prompt, bos=True), dtype=torch.long, device=device
        ).unsqueeze(0)
        if prompt_tensor.shape[1] >= student.config.max_sequence_length:
            raise ValueError("a prompt is too long for the student context window")
        generated = student.generate(
            prompt_ten

### 다른 선택지는 없나?

full estimator는 작은 vocab/저장 가능한 teacher logits에 적합하다. sampled estimator는 큰 vocab·API log-prob 환경에 유리하지만 여러 sample, control variate, clipping이 필요할 수 있다. old-policy rollout을 재사용하면 importance correction과 policy-version 감사가 필요하다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L05의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.algorithms import (collect_student_trajectories,
    on_policy_distillation_loss, score_teacher)
from opd_study.data import CharacterTokenizer, generate_tiny_arithmetic
from opd_study.models import TinyCausalLM, TinyTransformerConfig

tokenizer = CharacterTokenizer(); splits = generate_tiny_arithmetic(train_rows=4, validation_rows=1, test_rows=1)
config = TinyTransformerConfig(vocab_size=tokenizer.vocab_size, number_of_layers=1,
    hidden_size=32, number_of_heads=4, feed_forward_size=64)
student, teacher = TinyCausalLM(config), TinyCausalLM(config)
trajectories = collect_student_trajectories(student, [row.prompt for row in splits.train[:2]],
    tokenizer, max_new_tokens=4, min_new_tokens=4, temperature=0.0)
signals = score_teacher(teacher, trajectories)
student_logits = student(trajectories.token_ids, trajectories.attention_mask)
output = on_policy_distillation_loss(student_logits, trajectories, signals)
print("shapes:", trajectories.token_ids.shape, student_logits.shape, output.token_loss.shape)
print("response tokens:", int(trajectories.response_mask.sum()), "loss:", float(output.loss.detach()))

shapes: torch.Size([2, 40]) torch.Size([2, 40, 81]) torch.Size([2, 40])
response tokens: 8 loss: 0.011338572017848492


In [4]:
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3)
optimizer.zero_grad(); output.loss.backward(); optimizer.step()
print("teacher has gradients:", any(parameter.grad is not None for parameter in teacher.parameters()))
print("rollout snapshot detached:", not trajectories.student_logprobs.requires_grad)

teacher has gradients: False
rollout snapshot detached: True


## Checks

In [5]:
assert not trajectories.response_mask[:, :trajectories.prompt_lengths.min()].any()
assert output.loss.requires_grad
assert not any(parameter.grad is not None for parameter in teacher.parameters())
print("check passed: sample -> detached state -> teacher no_grad -> recomputed student update")

check passed: sample -> detached state -> teacher no_grad -> recomputed student update


**연습 (10분):** sampled reverse-KL의 advantage에서 `.detach()`를 제거했을 때 gradient 식에 생기는 추가 항을 적어보고, production 코드는 수정하지 말고 작은 복제 식으로 gradient 차이를 확인하라.

<details><summary>확인 기준</summary>detach가 없으면 advantage 자체의 student log-prob에도 gradient가 생겨 의도한 score-function estimator와 달라진다.</details>

## 내가 자주 틀리는 것

### M1 — rollout graph를 optimizer까지 유지하기

- 틀린 형태: discrete sampling을 통해 gradient가 흐른다고 기대한다.
- 왜 틀렸나: sampled token 선택은 미분 가능하지 않다.
- 고친 형태: trajectory를 detach하고 current logits를 재계산한다.
- 관련 검사: `test_rollout_snapshots_are_detached_and_mode_is_restored`

### M2 — prompt·padding까지 KL 평균에 넣기

- 틀린 형태: `[B,T]` loss를 그대로 mean한다.
- 왜 틀렸나: prompt 길이와 padding이 budget을 왜곡한다.
- 고친 형태: shifted response mask로 `masked_mean`한다.
- 관련 검사: `test_sft_counts_only_response_targets`

## 60초 요약

1. student rollout부터 업데이트까지 잇는다
2. full/sampled estimator를 구분한다
3. teacher gradient를 차단한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`vopd`](https://arxiv.org/abs/2605.07865v1) · `2605.07865v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)